In [11]:
import polars as pl
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm, trange
from datasets import load_dataset
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup
import math, os, random, gc, copy

In [12]:
def load_data(days_to_test=7):
    ds = load_dataset("yandex/yambda", data_dir="flat/5b", data_files="likes.parquet")
    data = ds['train'].to_polars().sort(['uid','timestamp'])
    data = data.with_columns(
        (pl.col("timestamp") / 86_400).cast(int).alias("days_from_start")
    )
    train_data = data.filter(
        pl.col('days_from_start') <= 300 - days_to_test
    )

    test_data = data.filter(
        pl.col('days_from_start') > 300 - days_to_test
    )
    
    del data
    gc.collect()

    train_users = train_data['uid'].unique().to_list()
    test_data = test_data.filter(
        pl.col('uid').is_in(train_users)
    )
    test_users = test_data['uid'].unique()
    test_data = test_data.group_by('uid').agg(
        pl.col('item_id').alias('items')
    )
    return train_data, test_data, test_users

In [13]:
def split_eval(data, days_to_eval=7):
    train_data = data.filter(
        pl.col('days_from_start') <= pl.col('days_from_start').max() - days_to_eval
    )
    
    eval_data = data.filter(
        pl.col('days_from_start') > pl.col('days_from_start').max() - days_to_eval
    )
    del data
    gc.collect()
    
    eval_users = eval_data['uid'].unique()
    eval_data = eval_data.group_by('uid').agg(
        pl.col('item_id').alias('items')
    )
    return train_data, eval_data, eval_users

In [14]:
import torch
from torch.utils.data import Dataset
import polars as pl

class TrainSeqDataset(Dataset):
    def __init__(self, data, items_mapping, max_len=256, share_memory=True):
        self.max_len = int(max_len)
        self.seq_len = self.max_len - 1
        self.num_items = len(items_mapping)
        seqs = (
            data.group_by("uid")
                .agg(pl.col("item_id"))
                .get_column("item_id")
                .to_list()
        )

        n = len(seqs)
        seq_tensor = torch.zeros((n, self.max_len), dtype=torch.long)
        for i, s in enumerate(seqs):
            s = s[:self.max_len]
            if len(s) == 0:
                continue
            mapped = [items_mapping[x] + 1 for x in s]
            seq_tensor[i, :len(mapped)] = torch.tensor(mapped, dtype=torch.long)
        if share_memory:
            seq_tensor.share_memory_()
        self.seq_tensor = seq_tensor

    def __len__(self):
        return self.seq_tensor.size(0)

    def __getitem__(self, idx):
        seq = self.seq_tensor[idx]
        inputs = seq[:-1]
        labels = seq[1:]
        mask = (labels != 0).float()

        negs = torch.randint(
            1, self.num_items + 1, (self.seq_len,), dtype=torch.long
        )

        eq = (negs == labels) & (labels != 0)
        if eq.any():
            negs[eq] = (negs[eq] % self.num_items) + 1

        return {
            "inputs": inputs,
            "labels": labels,
            "negatives": negs,
            "mask": mask,
        }

class PredictSeqDataset(Dataset):
    def __init__(
        self,
        data,
        items_mapping,
        max_len=256,
        uids=None,
        take_last=True,
        return_uid=False,
        share_memory=True
    ):
        self.items_mapping = items_mapping
        self.max_len = int(max_len)
        self.seq_len = self.max_len - 1
        self.return_uid = return_uid

        if "items" in data.columns:
            df = data.select(["uid", "items"])
        else:
            df = data.group_by("uid").agg(pl.col("item_id").alias("items"))

        if uids is not None:
            df_u = pl.DataFrame({"uid": list(uids)})
            df = df_u.join(df, on="uid", how="left")
        else:
            df = df.sort("uid")

        self.uids = df.get_column("uid").to_list()
        seqs = df.get_column("items").to_list()
        n = len(seqs)

        inputs = torch.zeros((n, self.seq_len), dtype=torch.long)

        for i, s in enumerate(seqs):
            if s is None or len(s) == 0:
                continue

            if take_last and len(s) > self.seq_len:
                s = s[-self.seq_len:]
            else:
                s = s[:self.seq_len]

            mapped = [items_mapping[x] + 1 for x in s]  # 0 = PAD
            inputs[i, :len(mapped)] = torch.tensor(mapped, dtype=torch.long)

        mask = inputs.ne(0)

        if share_memory:
            inputs.share_memory_()
            mask.share_memory_()

        self.inputs_tensor = inputs
        self.mask_tensor = mask

        if return_uid:
            try:
                uids_tensor = torch.tensor(self.uids, dtype=torch.long)
                if share_memory:
                    uids_tensor.share_memory_()
                self.uids_tensor = uids_tensor
            except Exception:
                self.uids_tensor = None

    def __len__(self):
        return self.inputs_tensor.size(0)

    def __getitem__(self, idx):
        out = {
            "inputs": self.inputs_tensor[idx],
            "mask": self.mask_tensor[idx],
        }
        if self.return_uid:
            if getattr(self, "uids_tensor", None) is not None:
                out["uid"] = self.uids_tensor[idx]
            else:
                out["uid"] = self.uids[idx]
        return out

In [15]:
def _dedup_preserve_order(seq):
    seen = set()
    out = []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def evaluate(
    gt_df,
    pred_df,
    ks=[10, 50, 100],
    uid_col="uid",
    items_col="items",
    eval_on="gt",
    skip_empty_gt=True,
    dedup_pred=True,
):
    ks = sorted(set(int(k) for k in ks))
    max_k = max(ks)

    gt_map = {}
    for row in gt_df.select([uid_col, items_col]).iter_rows(named=True):
        uid = row[uid_col]
        items = row[items_col] if row[items_col] is not None else []
        gt_map[uid] = list(items)

    pred_map = {}
    for row in pred_df.select([uid_col, items_col]).iter_rows(named=True):
        uid = row[uid_col]
        items = row[items_col] if row[items_col] is not None else []
        pred_map[uid] = list(items)

    if eval_on == "gt":
        uids = list(gt_map.keys())
    elif eval_on == "union":
        uids = list(set(gt_map.keys()) | set(pred_map.keys()))

    sum_recall = {k: 0.0 for k in ks}
    sum_ndcg = {k: 0.0 for k in ks}
    n_users = 0

    inv_logs = [1.0 / math.log2(i + 2) for i in range(max_k)]

    for uid in uids:
        gt_items = gt_map.get(uid, [])
        gt_set = set(gt_items)

        if len(gt_set) == 0:
            if skip_empty_gt:
                continue
            for k in ks:
                sum_recall[k] += 0.0
                sum_ndcg[k] += 0.0
            n_users += 1
            continue

        pred_items = pred_map.get(uid, [])
        if dedup_pred:
            pred_items = _dedup_preserve_order(pred_items)
        pred_top = pred_items[:max_k]

        cum_hits = []
        cum_dcg = []
        hits = 0
        dcg = 0.0
        for i, item in enumerate(pred_top):
            rel = 1 if item in gt_set else 0
            hits += rel
            dcg += rel * inv_logs[i]
            cum_hits.append(hits)
            cum_dcg.append(dcg)

        def _get_prefix(arr, k):
            idx = min(k, len(arr)) - 1
            return arr[idx] if idx >= 0 else 0

        for k in ks:
            hit_k = _get_prefix(cum_hits, k)
            dcg_k = float(_get_prefix(cum_dcg, k))

            recall_k = hit_k / len(gt_set)

            ideal_len = min(k, len(gt_set))
            idcg_k = sum(inv_logs[i] for i in range(ideal_len))
            ndcg_k = (dcg_k / idcg_k) if idcg_k > 0 else 0.0

            sum_recall[k] += recall_k
            sum_ndcg[k] += ndcg_k

        n_users += 1

    row = {}
    for k in ks:
        row[f"recall@{k}"] = (sum_recall[k] / n_users) if n_users else 0.0
        row[f"ndcg@{k}"] = (sum_ndcg[k] / n_users) if n_users else 0.0

    return row

In [16]:
class SASRec(nn.Module):
    def __init__(
        self,
        num_items,
        emb_dim=128,
        max_seq_len=256,
        num_layers=2,
        num_heads=4,
        dropout=0.0
    ):
        super().__init__()
        self.item_embeddings = nn.Embedding(num_items + 1,emb_dim,)
        self.position_embeddings = nn.Embedding(max_seq_len, emb_dim)

        self.layernorm = nn.LayerNorm(emb_dim)
        self.dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=num_heads,
            dim_feedforward=4 * emb_dim,
            dropout=dropout,
            activation=nn.GELU(),
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers)
        self._init_weights()
    
    @torch.no_grad()
    def _init_weights(self, initializer_range = 0.02):
        for key, value in self.named_parameters():
            if 'weight' in key:
                if 'norm' in key:
                    nn.init.ones_(value.data)
                else:
                    nn.init.trunc_normal_(
                        value.data, std=initializer_range, a=-2 * initializer_range, b=2 * initializer_range
                    )
            else:
                assert 'bias' in key
                nn.init.zeros_(value.data)

    def last_pooling(self, embeddings, mask):
        mask = mask.to(torch.bool)
        flatten_embeddings = embeddings[mask]
        lengths = torch.sum(mask, dim=-1)
        offsets = torch.cumsum(lengths, dim=0) 
        last_embeddings = flatten_embeddings[offsets.long() - 1]
        return last_embeddings

    def forward(self, inputs, attn_mask):
        embeds = self.item_embeddings(inputs)
        bs, num_embs, embs_dim = embeds.shape
        
        positions = torch.arange(start=0, end=num_embs, step=1, device=embeds.device)
        pos_embeds = self.position_embeddings(positions)
        embeds = embeds + pos_embeds
        embeds = self.dropout(self.layernorm(embeds))
        
        mask = torch.triu(torch.ones(num_embs, num_embs, device=embeds.device, dtype=torch.bool), diagonal=1)
        embeds = self.encoder(embeds, mask=mask)
        return embeds, self.last_pooling(embeds, attn_mask)

In [17]:
def set_random_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [23]:
class CFG:
    max_seq_len = 256
    train_bs = 256
    num_workers = 0
    eval_bs = 1024
    emb_dim = 128
    num_layers = 2
    num_heads = 4
    dropout = 0.0
    device = 'cuda'
    scheduler = True
    warmup_ratio = 0.1
    num_cycles = 0.5
    k = 100
    num_epoches = 5
    lr = 2e-3
    weight_decay = 0.0
    seed = 56

In [25]:
def train_model(cfg, train_data, eval_data, eval_users, items_mapping, items_inv_mapping):
    set_random_seed(cfg.seed)

    train_ds = TrainSeqDataset(train_data, items_mapping, max_len=cfg.max_seq_len)
    train_users = train_data['uid'].unique()
    eval_ds = PredictSeqDataset(train_data, items_mapping, uids = train_users, max_len=cfg.max_seq_len)
    
    train_dl = DataLoader(
        dataset = train_ds,
        batch_size = cfg.train_bs,
        shuffle=True,
        num_workers = cfg.num_workers,
        pin_memory = True
    )
    
    eval_dl = DataLoader(
        dataset = eval_ds,
        batch_size = cfg.eval_bs,
        shuffle=False,
        num_workers = cfg.num_workers,
        pin_memory = True
    )

    model = SASRec(
        num_items = len(items_mapping),
        emb_dim = cfg.emb_dim,
        max_seq_len = cfg.max_seq_len,
        num_layers = cfg.num_layers,
        num_heads = cfg.num_heads,
        dropout = cfg.dropout
    ).to(cfg.device)

    optimizer = AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    if cfg.scheduler:
        num_steps = len(train_dl) * cfg.num_epoches
        scheduler = get_cosine_schedule_with_warmup(
            optimizer = optimizer,
            num_training_steps = num_steps,
            num_warmup_steps = num_steps * cfg.warmup_ratio,
            num_cycles = cfg.num_cycles
        )
        
    best_metric = -1e9
    best_weights = copy.deepcopy(model.state_dict())

    for epoch in trange(cfg.num_epoches):
        pbar = tqdm(train_dl)
        model.train()
        for batch in pbar:
            for k in batch:
                batch[k] = batch[k].to(cfg.device)
                
            optimizer.zero_grad(set_to_none=True)
            embeds, _ = model(batch['inputs'], batch['mask'])
            labels_embeds = model.item_embeddings(batch['labels'])
            negatives_embeds = model.item_embeddings(batch['negatives'])
            
            labels_embeds = labels_embeds.view(-1,cfg.emb_dim)
            negatives_embeds = negatives_embeds.view(-1,cfg.emb_dim)
            embeds_flatten = embeds.view(-1,cfg.emb_dim)
            mask_flatten = batch['mask'].view(-1)
            
            logits_pos = torch.einsum('bd,bd -> b',embeds_flatten, labels_embeds)
            logits_neg = torch.einsum('bd,bd -> b',embeds_flatten, negatives_embeds)

            loss_pos = torch.nn.functional.binary_cross_entropy_with_logits(
                logits_pos, torch.ones_like(logits_pos), reduction="none"
            )
            loss_neg = torch.nn.functional.binary_cross_entropy_with_logits(
                logits_neg, torch.zeros_like(logits_neg), reduction="none"
            )
            
            m = mask_flatten.float()
            den = m.sum().clamp_min(1.0)
            loss = ((loss_pos + loss_neg) * m).sum() / den
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            pbar.set_postfix(loss=f"{loss.item():.4f}")
            if cfg.scheduler:
                scheduler.step()
                
        model.eval()
        predicted_embeds = []
        for batch in tqdm(eval_dl):
            for k in batch:
                batch[k] = batch[k].to(cfg.device)
            with torch.no_grad():
                _, embeds = model(batch['inputs'], batch['mask'])
            predicted_embeds.append(embeds.detach().cpu().numpy())
            
        predicted_embeds = np.concatenate(predicted_embeds)
        item_matrix = model.item_embeddings.weight.data.cuda()
        predicted_embeds = torch.from_numpy(predicted_embeds).cuda()

        bs_search = 128
        I_parts = []
        with torch.no_grad():
            for start in trange(0, predicted_embeds.shape[0], bs_search):
                end = min(start + bs_search, predicted_embeds.shape[0])
                q = predicted_embeds[start:end]
        
                scores = q @ item_matrix.T
                topk_idx = torch.topk(scores, k=cfg.k, dim=1).indices
                I_parts.append(topk_idx.cpu().numpy().astype(np.int64))
        
        I = np.concatenate(I_parts, axis=0)
        predicted_items = [[items_inv_mapping[max(0,int(x) - 1)] for x in seq] for seq in I]
        preds_df = pl.DataFrame({'uid':train_users, 'items':predicted_items})
        preds_df = preds_df.filter(
            pl.col('uid').is_in(eval_users)
        )
        metrics = evaluate(eval_data, preds_df)
        out = f'Epoch: {epoch} '
        for k in metrics:
            out += f'{k}: {metrics[k]} '
        print(out)
        metric = metrics['recall@100']
        if metric > best_metric:
            best_metric = metric
            best_weights = copy.deepcopy(model.state_dict())
            
    model.load_state_dict(best_weights)
    return model, best_metric

In [38]:
def predict_sasrec(model, cfg, train_all_data, test_users, items_mapping):
    train_users = train_all_data['uid'].unique()
    test_ds = PredictSeqDataset(train_all_data, items_mapping, uids = train_users, max_len=cfg.max_seq_len)
    test_dl = DataLoader(
        dataset = test_ds,
        batch_size = cfg.eval_bs,
        shuffle=False,
        num_workers = cfg.num_workers,
        pin_memory = True
    )

    user_embeds = []
    for batch in tqdm(test_dl):
        for k in batch:
            batch[k] = batch[k].to(cfg.device)
        with torch.no_grad():
            _, embeds = model(batch['inputs'], batch['mask'])
        user_embeds.append(embeds.detach().cpu().numpy())
            
    predicted_embeds = np.concatenate(user_embeds)
    item_matrix = model.item_embeddings.weight.data.cuda()
    predicted_embeds = torch.from_numpy(predicted_embeds).cuda()

    bs_search = 128
    I_parts = []
    with torch.no_grad():
        for start in trange(0, predicted_embeds.shape[0], bs_search):
            end = min(start + bs_search, predicted_embeds.shape[0])
            q = predicted_embeds[start:end]
        
            scores = q @ item_matrix.T
            topk_idx = torch.topk(scores, k=cfg.k, dim=1).indices
            I_parts.append(topk_idx.cpu().numpy().astype(np.int64))
        
    I = np.concatenate(I_parts, axis=0)
    predicted_items = [[items_inv_mapping[max(0,int(x) - 1)] for x in seq] for seq in I]
    preds_df = pl.DataFrame({'uid':train_users, 'items':predicted_items})
    preds_df = preds_df.filter(
            pl.col('uid').is_in(test_users)
    )
    return preds_df

In [26]:
all_train_data, test_data, test_users = load_data()
train_data, eval_data, eval_users = split_eval(all_train_data)

In [27]:
users_inv_mapping = dict(enumerate(all_train_data['uid'].unique()))
users_mapping = {v: k for k, v in users_inv_mapping.items()}

items_inv_mapping = dict(enumerate(all_train_data['item_id'].unique()))
items_mapping = {v: k for k, v in items_inv_mapping.items()}
len(users_mapping),len(items_mapping)

(820637, 1977318)

In [28]:
cfg = CFG()
model, _ = train_model(cfg, train_data, eval_data, eval_users, items_mapping, items_inv_mapping)

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/3173 [00:00<?, ?it/s]

  0%|          | 0/794 [00:00<?, ?it/s]

  0%|          | 0/6345 [00:00<?, ?it/s]

Epoch: 0 recall@10: 0.007968923810797768 ndcg@10: 0.006879640534922791 recall@50: 0.028515313941400406 ndcg@50: 0.012793415616506145 recall@100: 0.04834237947100459 ndcg@100: 0.017786629346858294 


...

Epoch: 3 recall@10: 0.014540710012593328 ndcg@10: 0.012052362988910324 recall@50: 0.049349466303799744 ndcg@50: 0.022257833403268142 recall@100: 0.0809561203056438 ndcg@100: 0.030222476741780083 


  0%|          | 0/3173 [00:00<?, ?it/s]

  0%|          | 0/794 [00:00<?, ?it/s]

  0%|          | 0/6345 [00:00<?, ?it/s]

Epoch: 4 recall@10: 0.01451997210803325 ndcg@10: 0.012043387905136167 recall@50: 0.0493225172944566 ndcg@50: 0.022292106693105034 recall@100: 0.08115873187982278 ndcg@100: 0.030327830342211922 


In [ ]:
sasrec_preds = predict_sasrec(model, cfg, all_train_data, test_users, items_mapping)

  0%|          | 0/802 [00:00<?, ?it/s]

In [41]:
evaluate(test_data, sasrec_preds)

{'recall@10': 0.0131113763886066,
 'ndcg@10': 0.010764798346247753,
 'recall@50': 0.04511022266302349,
 'ndcg@50': 0.020134211240563946,
 'recall@100': 0.07444323607747394,
 'ndcg@100': 0.027489170499836884}